# Supervised Learning 1: Übungen

Eigene Übungen zum Stoff von Supervised Learning, Tag 1: Gradient Descent, Fehlermasse, Train/Test, Cross-Validation, Imputation, One-Hot, Overfitting, Regularisierung, Hyperparameter-Suche, Feature Selection.

Alle Aufgaben nutzen den eingebauten Datensatz `load_diabetes` von scikit-learn, du brauchst keine Downloads. Zufallszahlen sind mit Seed 42 fixiert.

**Ablauf:** Zelle für Zelle ausführen, `...` ersetzen, Prüfzelle laufen lassen, danach die Lösung ansehen.

Benötigt: `pandas`, `numpy`, `matplotlib`, `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def ok(condition, hint=""):
    """Print a success message or raise a readable error."""
    assert condition, "❌ " + hint
    print("✅ Richtig!")

from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer

# Bundled dataset (works offline): 442 patients, 10 features, target = disease progression
X, y = load_diabetes(return_X_y=True, as_frame=True)
SEED = 42
X.head()

### Aufgabe 1: Gradient Descent von Hand

Minimiere `f(x) = x²` mit Gradient Descent. Die Steigung ist `2 * x`. Ergänze die Update-Regel `x = x - lernrate * steigung`.

Danach: Teste die Lernraten 0.2, 1.0 und 1.1 und beobachte, was passiert.

In [ ]:
def gradient_descent(lr, x0=-4.0, steps=50):
    x = x0
    for _ in range(steps):
        x = ...   # update rule
    return x


for lr in [0.2, 1.0, 1.1]:
    print(lr, gradient_descent(lr))

In [ ]:
ok(abs(gradient_descent(0.2)) < 1e-6 and abs(abs(gradient_descent(1.0)) - 4) < 1e-9 and abs(gradient_descent(1.1)) > 100,
   "Lernrate 0.2 soll gegen 0 laufen, 1.0 springt hin und her, 1.1 divergiert.")

<details>
<summary>Lösung anzeigen</summary>

```python
def gradient_descent(lr, x0=-4.0, steps=50):
    x = x0
    for _ in range(steps):
        x = x - lr * 2 * x
    return x


for lr in [0.2, 1.0, 1.1]:
    print(lr, gradient_descent(lr))
```

</details>

### Aufgabe 2: Fehlermasse von Hand

Berechne MAE, MSE und R² für `y_true` und `y_pred` **ohne** sklearn, nur mit NumPy. Vergleiche danach mit sklearn.

In [ ]:
y_true = np.array([3, 5, 2, 7])
y_pred = np.array([2.5, 5, 4, 8])

residuen = ...
mae = ...
mse = ...
r2 = ...   # 1 - ss_res / ss_tot

In [ ]:
ok(np.isclose(mae, mean_absolute_error(y_true, y_pred)) and np.isclose(mse, mean_squared_error(y_true, y_pred))
   and np.isclose(r2, r2_score(y_true, y_pred)),
   "MAE = Mittel der Beträge, MSE = Mittel der Quadrate, R² = 1 - ss_res / ss_tot.")
print(f"MAE={mae:.3f}  MSE={mse:.3f}  R2={r2:.3f}")

<details>
<summary>Lösung anzeigen</summary>

```python
y_true = np.array([3, 5, 2, 7])
y_pred = np.array([2.5, 5, 4, 8])

residuen = y_true - y_pred
mae = np.mean(np.abs(residuen))
mse = np.mean(residuen ** 2)
ss_res = np.sum(residuen ** 2)
ss_tot = np.sum((y_true - y_true.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
```

</details>

### Aufgabe 3: Erstes Modell: Split und lineare Regression

Teile `X`, `y` in Training und Test (80/20, `random_state=SEED`), trainiere eine `LinearRegression` auf den Trainingsdaten und berechne R² auf den **Testdaten** (`r2_test`).

In [ ]:
X_train, X_test, y_train, y_test = ...
model = ...
r2_test = ...

In [ ]:
ok(len(X_test) == 89 and 0.3 < r2_test < 0.7,
   "test_size=0.2 ergibt 89 Testzeilen. R² auf dem Test-Set liegt bei diesem Datensatz etwa bei 0.45.")
print(f"R2 auf Testdaten: {r2_test:.3f}")

<details>
<summary>Lösung anzeigen</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
model = LinearRegression().fit(X_train, y_train)
r2_test = r2_score(y_test, model.predict(X_test))
```

</details>

### Aufgabe 4: Pipeline und Cross-Validation

Baue eine `Pipeline` aus `StandardScaler` und `LinearRegression` und bewerte sie mit 5-facher Cross-Validation (`scoring="r2"`) auf den **Trainingsdaten**. Gib Mittelwert und Standardabweichung aus.

In [ ]:
pipe = ...
scores = ...
print(scores.mean(), scores.std())

In [ ]:
ok(len(scores) == 5 and 0.3 < scores.mean() < 0.65,
   "cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2') liefert 5 Werte.")

<details>
<summary>Lösung anzeigen</summary>

```python
pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")
print(scores.mean(), scores.std())
```

</details>

### Aufgabe 5: Fehlende Werte auffüllen (Imputation)

In `X_missing` wurden 10 % der Werte der Spalte `bmi` gelöscht. Ergänze die Pipeline um einen `SimpleImputer(strategy="median")` **vor** dem Scaler und bewerte sie wieder mit 5-facher Cross-Validation.

In [ ]:
X_missing = X_train.copy()
rows = np.random.default_rng(SEED).choice(len(X_missing), size=len(X_missing) // 10, replace=False)
X_missing.iloc[rows, X_missing.columns.get_loc("bmi")] = np.nan

pipe_imp = ...
scores_imp = ...

In [ ]:
ok(X_missing["bmi"].isna().sum() > 0 and len(scores_imp) == 5 and np.isfinite(scores_imp).all() and scores_imp.mean() > 0.25,
   "Reihenfolge in der Pipeline: erst Imputer, dann Scaler, dann Modell.")
print(f"Mit Imputation: {scores_imp.mean():.3f}  (ohne fehlende Werte: {scores.mean():.3f})")

<details>
<summary>Lösung anzeigen</summary>

```python
X_missing = X_train.copy()
rows = np.random.default_rng(SEED).choice(len(X_missing), size=len(X_missing) // 10, replace=False)
X_missing.iloc[rows, X_missing.columns.get_loc("bmi")] = np.nan

pipe_imp = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
scores_imp = cross_val_score(pipe_imp, X_missing, y_train, cv=5, scoring="r2")
```

</details>

### Aufgabe 6: One-Hot-Encoding

Wandle die Spalte `stadt` mit One-Hot-Encoding um (`pd.get_dummies`). Warum ist es falsch, Städte einfach als 1, 2, 3 zu codieren?

In [ ]:
kunden = pd.DataFrame({"stadt": ["Luzern", "Zug", "Luzern", "Bern"], "umsatz": [10, 20, 30, 40]})
kunden_ohe = ...

In [ ]:
ok(kunden_ohe.shape == (4, 4) and {"stadt_Luzern", "stadt_Zug", "stadt_Bern"} <= set(kunden_ohe.columns),
   "get_dummies(df, columns=['stadt']) erzeugt eine 0/1-Spalte pro Stadt.")
kunden_ohe

<details>
<summary>Lösung anzeigen</summary>

```python
kunden = pd.DataFrame({"stadt": ["Luzern", "Zug", "Luzern", "Bern"], "umsatz": [10, 20, 30, 40]})
kunden_ohe = pd.get_dummies(kunden, columns=["stadt"])
```

</details>

### Aufgabe 7: Overfitting selbst erleben

Wir passen Polynome vom Grad 1 bis 10 an verrauschte Sinus-Daten an. Ergänze das Modell (`PolynomialFeatures` + `LinearRegression` in einer Pipeline) und berechne Trainings- und Testfehler (MSE) pro Grad. Bestimme `best_deg`, den Grad mit dem kleinsten **Testfehler**.

Beobachte: Was passiert mit dem Trainingsfehler, was mit dem Testfehler?

In [ ]:
rng2 = np.random.default_rng(0)
x = np.sort(rng2.uniform(-1, 1, 30))
yy = np.sin(2 * np.pi * x) + rng2.normal(0, 0.3, 30)
xt, xe, yt, ye = train_test_split(x.reshape(-1, 1), yy, test_size=0.5, random_state=0)

train_mse, test_mse = {}, {}
for deg in range(1, 11):
    m = ...   # Pipeline: PolynomialFeatures(deg, include_bias=False) + LinearRegression
    m.fit(xt, yt)
    train_mse[deg] = ...
    test_mse[deg] = ...

best_deg = ...
print(best_deg)

In [ ]:
ok(train_mse[10] < train_mse[1] and 2 <= best_deg <= 9 and test_mse[10] > test_mse[best_deg],
   "Trainingsfehler sinkt mit dem Grad immer. Der Testfehler hat sein Minimum in der Mitte.")
plt.plot(list(train_mse), list(train_mse.values()), label="Training")
plt.plot(list(test_mse), list(test_mse.values()), label="Test")
plt.yscale("log"); plt.xlabel("Polynomgrad"); plt.ylabel("MSE"); plt.legend(); plt.show()

<details>
<summary>Lösung anzeigen</summary>

```python
rng2 = np.random.default_rng(0)
x = np.sort(rng2.uniform(-1, 1, 30))
yy = np.sin(2 * np.pi * x) + rng2.normal(0, 0.3, 30)
xt, xe, yt, ye = train_test_split(x.reshape(-1, 1), yy, test_size=0.5, random_state=0)

train_mse, test_mse = {}, {}
for deg in range(1, 11):
    m = Pipeline([
        ("poly", PolynomialFeatures(deg, include_bias=False)),
        ("model", LinearRegression()),
    ])
    m.fit(xt, yt)
    train_mse[deg] = mean_squared_error(yt, m.predict(xt))
    test_mse[deg] = mean_squared_error(ye, m.predict(xe))

best_deg = min(test_mse, key=test_mse.get)
print(best_deg)
```

</details>

### Aufgabe 8: Lasso streicht, Ridge schrumpft

Skaliere `X_train` (`StandardScaler`) und trainiere `Lasso` mit `alpha` in `[0.01, 0.1, 1.0]`. Zähle pro `alpha`, wie viele Koeffizienten **exakt 0** sind (`n_zero`). Vergleiche danach die Summe der Beträge der Koeffizienten von `Ridge(alpha=10)` und `LinearRegression`.

In [ ]:
Xs = StandardScaler().fit_transform(X_train)

n_zero = {}
for a in [0.01, 0.1, 1.0]:
    lasso = ...
    n_zero[a] = ...

ridge_sum = ...    # sum of absolute coefficients of Ridge(alpha=10)
linear_sum = ...   # same for LinearRegression
print(n_zero, ridge_sum, linear_sum)

In [ ]:
ok(n_zero[1.0] >= n_zero[0.1] >= n_zero[0.01] and n_zero[1.0] > 0 and ridge_sum < linear_sum,
   "Je grösser alpha, desto mehr Koeffizienten sind bei Lasso 0. Ridge macht die Koeffizienten insgesamt kleiner.")

<details>
<summary>Lösung anzeigen</summary>

```python
Xs = StandardScaler().fit_transform(X_train)

n_zero = {}
for a in [0.01, 0.1, 1.0]:
    lasso = Lasso(alpha=a, max_iter=10_000).fit(Xs, y_train)
    n_zero[a] = int(np.sum(lasso.coef_ == 0))

ridge_sum = np.abs(Ridge(alpha=10).fit(Xs, y_train).coef_).sum()
linear_sum = np.abs(LinearRegression().fit(Xs, y_train).coef_).sum()
print(n_zero, ridge_sum, linear_sum)
```

</details>

### Aufgabe 9: Hyperparameter-Suche mit GridSearchCV

Suche für eine Pipeline (`StandardScaler` + `Lasso`) das beste `alpha` aus `[0.01, 0.1, 1, 10]` mit 5-facher Cross-Validation auf den Trainingsdaten. Bewerte danach das beste Modell **einmal** auf den Testdaten (`r2_final`).

In [ ]:
pipe_l = Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10_000))])
grid = ...
search = ...
search.fit(X_train, y_train)
r2_final = ...
print(search.best_params_, r2_final)

In [ ]:
ok(hasattr(search, "best_params_") and "model__alpha" in search.best_params_ and 0.3 < r2_final < 0.7,
   "Der Parametername lautet schritt__parameter, also model__alpha.")

<details>
<summary>Lösung anzeigen</summary>

```python
pipe_l = Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10_000))])
grid = {"model__alpha": [0.01, 0.1, 1, 10]}
search = GridSearchCV(pipe_l, grid, cv=5, scoring="r2")
search.fit(X_train, y_train)
r2_final = r2_score(y_test, search.best_estimator_.predict(X_test))
print(search.best_params_, r2_final)
```

</details>

### Aufgabe 10: Feature Selection mit Korrelation

Berechne die absolute Korrelation jedes Merkmals mit dem Ziel (`X_train.corrwith(y_train).abs()`) und wähle alle Merkmale mit Korrelation **über 0.3** (`gewaehlt`, Liste der Namen). Vergleiche dann die 5-fache Cross-Validation mit allen und mit den gewählten Merkmalen.

Frage: Wird das Modell mit weniger Merkmalen besser, schlechter oder gleich gut? Was bedeutet das für die Erklärbarkeit?

In [ ]:
korr = ...
gewaehlt = ...

pipe_fs = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
alle = cross_val_score(pipe_fs, X_train, y_train, cv=5, scoring="r2").mean()
auswahl = ...
print(len(gewaehlt), alle, auswahl)

In [ ]:
ok(len(gewaehlt) >= 4 and "bmi" in gewaehlt and 0.2 < auswahl < 0.7,
   "corrwith(y).abs(), dann mit einer Maske > 0.3 filtern und die Namen als Liste nehmen.")

<details>
<summary>Lösung anzeigen</summary>

```python
korr = X_train.corrwith(y_train).abs()
gewaehlt = korr[korr > 0.3].index.tolist()

pipe_fs = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
alle = cross_val_score(pipe_fs, X_train, y_train, cv=5, scoring="r2").mean()
auswahl = cross_val_score(pipe_fs, X_train[gewaehlt], y_train, cv=5, scoring="r2").mean()
print(len(gewaehlt), alle, auswahl)
```

</details>